# Simple Agents, Tools, and MCP Servers

A RAG pipeline runs the same steps for every input. An **agent** chooses its
steps: the model decides which tool to call, reads the result, and decides
again. This session builds that loop from first principles, then shows how
the Model Context Protocol (MCP) standardizes the tool side so any host can
talk to any tool server.

## Tools are just functions with contracts

An agent's tools need a name, a description the model can reason over, and
a callable. That triple *is* the tool interface — every framework dresses up
exactly this.

In [1]:
# The knowledge base for every lab this week: support documents for Atlas
# Cycles, a fictional e-bike maker. Small enough to read, real enough to
# retrieve against.
CORPUS = {
    "battery-care": (
        "Atlas S2 battery care. Charge the battery to 80 percent for daily "
        "use and only to 100 percent before a long ride. Store between 10 "
        "and 25 degrees Celsius. A full recharge takes 4.5 hours from empty."
    ),
    "warranty": (
        "Atlas warranty policy. The frame is covered for 5 years. The "
        "battery and motor are covered for 2 years or 15,000 km, whichever "
        "comes first. Wear parts such as brake pads and tires are excluded."
    ),
    "range": (
        "Atlas S2 range guide. Expect 90 to 110 km in Eco mode, 60 to 75 km "
        "in Trail mode, and 40 to 55 km in Boost mode. Headwind, cargo "
        "weight, and cold weather reduce range by up to 30 percent."
    ),
    "error-codes": (
        "Atlas display error codes. E01 means a motor sensor fault: restart "
        "the system. E04 means battery communication lost: reseat the "
        "battery. E09 means brake cutoff engaged: check the brake levers."
    ),
    "first-service": (
        "First service. Book the complimentary first service after 300 km "
        "or 3 months. Spoke tension, brake bedding, and firmware updates "
        "are included at no charge."
    ),
}

# A deterministic embedding: character trigram counts. No model, no network,
# yet it captures enough word-shape overlap to demonstrate the geometry that
# real embedding models learn.
from collections import Counter
import math

def embed(text):
    t = " " + "".join(c.lower() if c.isalnum() else " " for c in text) + " "
    return Counter(t[i:i+3] for i in range(len(t) - 2) if t[i:i+3].strip())

def cosine(a, b):
    dot = sum(a[k] * b[k] for k in a.keys() & b.keys())
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

import ast, operator as op

_OPS = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv}

def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    raise ValueError("unsupported expression")

def calculator(expression):
    # Arithmetic only — never eval() model output.
    return str(_safe_eval(ast.parse(expression, mode="eval").body))

DOC_VECTORS = {name: embed(doc) for name, doc in CORPUS.items()}

def kb_search(query):
    best = max(DOC_VECTORS, key=lambda n: cosine(embed(query), DOC_VECTORS[n]))
    return f"[{best}] {CORPUS[best]}"

TOOLS = {
    "calculator": ("Evaluate an arithmetic expression, e.g. 18*4.", calculator),
    "kb_search": ("Search the Atlas Cycles knowledge base.", kb_search),
}
for name, (desc, _) in TOOLS.items():
    print(f"{name:<12} {desc}")

calculator   Evaluate an arithmetic expression, e.g. 18*4.
kb_search    Search the Atlas Cycles knowledge base.


## The policy

In production the *model* decides the next action by reading the transcript.
Here a scripted `DemoPolicy` makes the same decisions deterministically —
the loop around it is identical either way, which is the point: the loop is
infrastructure you own; the policy is the part you rent from a model.

In [2]:
class DemoPolicy:
    # Stands in for the LLM's action choice. Reads the transcript, returns
    # the next action as (tool, argument) or ("finish", answer).
    def decide(self, question, transcript):
        import re
        math_expr = re.search(r"\d+\s*[-+*/]\s*\d+", question)
        if math_expr and not any(t[0] == "calculator" for t in transcript):
            return "calculator", math_expr.group(0).replace(" ", "")
        if not any(t[0] == "kb_search" for t in transcript):
            return "kb_search", question
        facts = "; ".join(obs for _, _, obs in transcript)
        return "finish", f"Combining what I found: {facts}"

policy = DemoPolicy()
print(policy.decide("what is 6*7", []))

('calculator', '6*7')


## The agent loop

Thought → Action → Observation, until the policy says finish. A step cap
guards against loops — the first lesson of running agents in production.

In [3]:
def run_agent(question, max_steps=5):
    transcript = []
    for step in range(1, max_steps + 1):
        action, arg = policy.decide(question, transcript)
        if action == "finish":
            print(f"step {step}: FINISH")
            return arg
        _, fn = TOOLS[action]
        observation = fn(arg)
        transcript.append((action, arg, observation))
        print(f"step {step}: {action}({arg!r})")
        print(f"         -> {observation[:70]}")
    return "step budget exhausted"

answer = run_agent(
    "How long is the Atlas battery warranty, and what is 18*4?")
print("\nANSWER:", answer[:120], "...")

step 1: calculator('18*4')
         -> 72
step 2: kb_search('How long is the Atlas battery warranty, and what is 18*4?')
         -> [warranty] Atlas warranty policy. The frame is covered for 5 years. Th
step 3: FINISH

ANSWER: Combining what I found: 72; [warranty] Atlas warranty policy. The frame is covered for 5 years. The battery and motor ar ...


## MCP: the USB port for tools

Every agent framework reinvented tool registration until MCP standardized
it: a **host** (your app) runs a **client** that speaks JSON-RPC to a tool
**server**. The server advertises `tools/list` and executes `tools/call` —
so a tool written once works in any MCP-speaking host.

Below is that exchange, in-process and unadorned, so you can see exactly
what goes over the wire.

In [4]:
import json

class MiniMCPServer:
    # The two methods at the heart of the protocol, in-process.
    def handle(self, request):
        method, params = request["method"], request.get("params", {})
        if method == "tools/list":
            result = {"tools": [
                {"name": n, "description": d} for n, (d, _) in TOOLS.items()]}
        elif method == "tools/call":
            _, fn = TOOLS[params["name"]]
            result = {"content": [{"type": "text",
                                   "text": fn(params["argument"])}]}
        else:
            return {"jsonrpc": "2.0", "id": request["id"],
                    "error": {"code": -32601, "message": "unknown method"}}
        return {"jsonrpc": "2.0", "id": request["id"], "result": result}

server = MiniMCPServer()
for req in [
    {"jsonrpc": "2.0", "id": 1, "method": "tools/list"},
    {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
     "params": {"name": "calculator", "argument": "18*4"}},
]:
    print(">>", json.dumps(req))
    print("<<", json.dumps(server.handle(req))[:100])
    print()

>> {"jsonrpc": "2.0", "id": 1, "method": "tools/list"}
<< {"jsonrpc": "2.0", "id": 1, "result": {"tools": [{"name": "calculator", "description": "Evaluate an 

>> {"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": "calculator", "argument": "18*4"}}
<< {"jsonrpc": "2.0", "id": 2, "result": {"content": [{"type": "text", "text": "72"}]}}



Real MCP adds transports (stdio, HTTP), capability negotiation, resources,
and prompts — but the shape you just saw is the whole idea: **discovery,
then invocation, over a standard envelope.**

**Takeaways**
- An agent = a loop you own + a policy you rent + tools with contracts.
- Cap the steps. Log every action. You will need both by Day 3.
- MCP turns "integrate N tools with M hosts" from N×M work into N+M.